In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
import time
import os

class ActorCritic(nn.Module):
    def __init__(self, input_dim, action_dim, hidden_dim=128):
        super(ActorCritic, self).__init__()
        # Actor
        self.actor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
            nn.Softmax(dim=-1)
        )
        # Critic
        self.critic = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, state):
        action_probs = self.actor(state)
        state_value = self.critic(state)
        return action_probs, state_value

class CargoEnv:
    def __init__(self, grid_size=3,
                 start_file='data_graphform/3x3/tar3/start/graph_start_3x3_tar3_sp2_0.txt',
                 end_file='data_graphform/3x3/tar3/end/graph_end_3x3_tar3_sp2_0.txt'):
        self.grid_size = grid_size

        self.agents_with_targets_positions, self.agents_without_targets_positions, self.agent_ids = self.read_positions_from_file(start_file)
        self.num_agents_with_targets = len(self.agents_with_targets_positions)
        self.num_agents_without_targets = len(self.agents_without_targets_positions)
        self.num_agents = self.num_agents_with_targets + self.num_agents_without_targets
        self.targets = self.read_targets_from_file(end_file)
        self.positions = np.vstack((self.agents_with_targets_positions, self.agents_without_targets_positions))
        self.first_reach = [False] * self.num_agents_with_targets

        print("Agents with targets positions:", self.agents_with_targets_positions)
        print("Agents without targets positions:", self.agents_without_targets_positions)
        print("Total number of agents with targets:", self.num_agents_with_targets)
        print("Total number of agents without targets:", self.num_agents_without_targets)
        print("Total number of agents:", self.num_agents)
        print("Agent IDs:", self.agent_ids)

    def read_positions_from_file(self, file_path):
        agents_with_targets_positions = []
        agents_without_targets_positions = []
        agent_ids = []
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"文件未找到：{file_path}")
        with open(file_path, 'r') as f:
            lines = f.readlines()[1:]  # 忽略第一行(如39)
            for i, line in enumerate(lines):
                line = line.replace(',', ' ').replace('\t', ' ')
                numbers = [int(num) for num in line.strip().split()]
                for j, value in enumerate(numbers):
                    if value > 0:
                        agents_with_targets_positions.append([i, j])
                        agent_ids.append(value)
                    elif value < 0:
                        agents_without_targets_positions.append([i, j])
                    # value == 0為空格，不加入
        return np.array(agents_with_targets_positions), np.array(agents_without_targets_positions), agent_ids

    def read_targets_from_file(self, file_path):
        targets = [None] * len(self.agent_ids)
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"文件未找到：{file_path}")
        with open(file_path, 'r') as f:
            lines = f.readlines()[1:]  # 忽略第一行
            for i, line in enumerate(lines):
                line = line.replace(',', ' ').replace('\t', ' ')
                numbers = [int(num) for num in line.strip().split()]
                for j, value in enumerate(numbers):
                    if value > 0 and value in self.agent_ids:
                        idx = self.agent_ids.index(value)
                        targets[idx] = [i, j]
        if None in targets:
            raise ValueError("目標位置不完整，與貨物編號無法對應。")
        return np.array(targets)

    def reset(self):
        self.positions = np.vstack((self.agents_with_targets_positions, self.agents_without_targets_positions))
        self.first_reach = [False] * self.num_agents_with_targets
        return self.positions

    def step(self, actions):
        rewards = np.zeros(self.num_agents)
        planned_positions = []

        for agent_idx, action in enumerate(actions):
            if action is None:
                planned_positions.append(tuple(self.positions[agent_idx]))
                continue

            x, y = self.positions[agent_idx]
            if action == 0: # 上
                new_x, new_y = max(0, x - 1), y
            elif action == 1: # 下
                new_x, new_y = min(self.grid_size - 1, x + 1), y
            elif action == 2: # 左
                new_x, new_y = x, max(0, y - 1)
            elif action == 3: # 右
                new_x, new_y = x, min(self.grid_size - 1, y + 1)
            else:
                new_x, new_y = x, y
            planned_positions.append((new_x, new_y))

        final_positions = list(planned_positions)
        conflict = True
        max_attempts = 10
        attempts = 0
        while conflict and attempts < max_attempts:
            conflict = False
            position_counts = {}
            for pos in final_positions:
                position_counts[pos] = position_counts.get(pos, 0) + 1
            for idx, pos in enumerate(final_positions):
                if position_counts[pos] > 1:
                    conflict = True
                    final_positions[idx] = tuple(self.positions[idx])
                    actions[idx] = None
            attempts += 1

        for agent_idx in range(self.num_agents):
            old_position = self.positions[agent_idx]
            new_position = final_positions[agent_idx]
            self.positions[agent_idx] = new_position

            time_cost = -0.01

            if agent_idx < self.num_agents_with_targets:
                old_distance = np.linalg.norm(np.array(old_position) - self.targets[agent_idx])
                new_distance = np.linalg.norm(np.array(new_position) - self.targets[agent_idx])

                reward = 0
                if new_distance < old_distance:
                    reward += 50
                elif new_distance > old_distance:
                    reward -= 10
                if np.array_equal(new_position, self.targets[agent_idx]) and not self.first_reach[agent_idx]:
                    reward += 100
                    self.first_reach[agent_idx] = True
                reward += time_cost
                rewards[agent_idx] = reward
            else:
                # 無目標貨物：不動 -0.5，動 -0.1，加上時間成本
                if actions[agent_idx] is None:
                    rew = -0.5
                else:
                    rew = -0.1
                rew += time_cost
                rewards[agent_idx] = rew

        return self.positions, rewards

    def check_all_agents_at_target(self):
        return all(
            np.array_equal(self.positions[i], self.targets[i])
            for i in range(self.num_agents_with_targets)
        )

def get_grid_size(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"文件未找到：{file_path}")
    with open(file_path, 'r') as f:
        lines = f.readlines()
        map_lines = lines[1:]  # 忽略第一行
        grid_size = len([line for line in map_lines if line.strip()])
    return grid_size

def select_action(model, state, epsilon, valid_actions, stuck_or_done, all_actions=[0,1,2,3]):
    if stuck_or_done or not valid_actions:
        valid_actions = all_actions

    action_probs, state_value = model(state)
    action_probs = action_probs[0]

    if random.random() < epsilon:
        action = random.choice(valid_actions)
    else:
        probs = action_probs.clone()
        mask = torch.ones(probs.size(0), dtype=torch.bool)
        for a in range(probs.size(0)):
            if a not in valid_actions:
                mask[a] = False
        valid_probs = probs[mask]

        # 防止出現全0或sum=0的情況
        sum_valid = valid_probs.sum().item()
        if sum_valid <= 0 or torch.isnan(valid_probs).any():
            # 如果沒有有效機率或出現NaN，則隨機選擇
            action = random.choice(valid_actions)
        else:
            valid_probs = valid_probs / sum_valid
            valid_indices = [a for a in range(probs.size(0)) if a in valid_actions]
            chosen = np.random.choice(len(valid_indices), p=valid_probs.detach().cpu().numpy())
            action = valid_indices[chosen]

    action_prob = action_probs[action]
    log_prob = torch.log(action_prob)
    return action, log_prob, state_value

def main():
    start_time = time.time()
    start_file = 'data_graphform/4x4/tar3/start/graph_start_4x4_tar3_sp3_0.txt'
    end_file = 'data_graphform/4x4/tar3/end/graph_end_4x4_tar3_sp3_0.txt'

    grid_size = get_grid_size(start_file)
    env = CargoEnv(grid_size=grid_size, start_file=start_file, end_file=end_file)

    num_agents = env.num_agents
    num_agents_with_targets = env.num_agents_with_targets
    input_dim = 2 + 2 + (num_agents - 1) * 2
    print(f"Input dimension: {input_dim}")

    model = ActorCritic(input_dim=input_dim, action_dim=4)
    print("Model architecture:")
    print(model)

    optimizer = optim.Adam(model.parameters(), lr=0.0005)
    gamma = 0.99
    epsilon = 1.0
    epsilon_decay = 0.995
    epsilon_min = 0.01

    episodes = 10000
    max_steps = 50

    best_episode = None
    best_reward = -float('inf')
    best_paths = None

    for episode in range(episodes):
        obs = env.reset()
        done = False
        step_count = 0

        log_probs = []
        values = []
        rewards_list = []
        paths = [[] for _ in range(num_agents)]
        global_time_step = 0

        while not done and step_count < max_steps:
            step_count += 1
            actions = []
            occupied_positions = set()

            for pos in env.positions:
                occupied_positions.add(tuple(pos))

            moves = {0:(-1,0),1:(1,0),2:(0,-1),3:(0,1)}

            for i in range(num_agents):
                own_position = env.positions[i]
                if i < num_agents_with_targets:
                    own_target = env.targets[i]
                else:
                    own_target = np.array([0,0])
                other_positions = np.delete(env.positions, i, axis=0).flatten()
                state_arr = np.concatenate((own_position, own_target, other_positions))
                state_tensor = torch.tensor(state_arr, dtype=torch.float32, requires_grad=True).unsqueeze(0)

                valid_actions = []
                x, y = env.positions[i]
                for a,(dx,dy) in moves.items():
                    nx, ny = x+dx,y+dy
                    if 0<=nx<env.grid_size and 0<=ny<env.grid_size:
                        if (nx,ny) not in occupied_positions:
                            valid_actions.append(a)
                stuck_or_done = not valid_actions

                action, log_prob, state_value = select_action(model, state_tensor, epsilon, valid_actions, stuck_or_done)
                actions.append(action)

                if i < num_agents_with_targets:
                    log_probs.append(log_prob.unsqueeze(0))
                    values.append(state_value.unsqueeze(0))

            new_positions, step_rewards = env.step(actions)
            global_time_step += 1

            for i in range(num_agents):
                paths[i].append((global_time_step, tuple(env.positions[i])))

            for i in range(num_agents_with_targets):
                rewards_list.append(step_rewards[i])

            done = env.check_all_agents_at_target()

        if len(rewards_list) > 0:
            returns = []
            R = 0
            for r in reversed(rewards_list):
                R = r + gamma*R
                returns.insert(0,R)
            returns = torch.tensor(returns, dtype=torch.float32)
            if len(values) > 0:
                values_t = torch.cat(values)
                log_probs_t = torch.cat(log_probs)

                advantages = returns - values_t.detach()
                actor_loss = -(log_probs_t * advantages).mean()
                critic_loss = 0.5 * (advantages.pow(2)).mean()

                loss = actor_loss + critic_loss
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                optimizer.step()

        total_episode_reward = sum(rewards_list)
        if total_episode_reward > best_reward:
            best_reward = total_episode_reward
            best_episode = episode + 1
            best_paths = paths

        epsilon = max(epsilon_min, epsilon * epsilon_decay)

    end_time = time.time()
    print(f"Total training time: {end_time - start_time:.4f} seconds")
    print(f"\nBest Episode: {best_episode} with Reward: {best_reward}")
    for i in range(num_agents):
        agent_type = "Target" if i < num_agents_with_targets else "Obstacle"
        print(f"Best Path for Agent {i + 1} ({agent_type}):")
        for time_step, position in best_paths[i]:
            print(f"  Time {time_step}: Position {position}")

    max_time = 0
    for i in range(num_agents):
        if len(best_paths[i]) > 0:
            max_time = max(max_time, best_paths[i][-1][0])

    print("\n最佳路徑以網格形式顯示：")
    for t in range(1, max_time + 1):
        grid = [["0" for _ in range(env.grid_size)] for _ in range(env.grid_size)]
        for i in range(num_agents):
            pos_record = [pos for (time_step, pos) in best_paths[i] if time_step == t]
            if len(pos_record) > 0:
                x, y = pos_record[0]
                if i < num_agents_with_targets:
                    grid[x][y] = str(i + 1)
                else:
                    grid[x][y] = "X"
        print("----------------------------------------")
        print(f"time period {t}")
        for row in grid:
            formatted_row = " ".join(f"{cell:2}" for cell in row)
            print(f"| {formatted_row} |")

if __name__ == "__main__":
    main()
